In [1]:
import utils

# Ensure ffmpeg is installed



In [2]:
upload_id = "003"
upload_dir = "/mnt/g/uploads/"
models_dir = "/mnt/g/models/"
artifacts_dir = "/mnt/g/song_artifacts/"
model_filename = "0.pth"
# TODO check and if not present make folders
# songslice
# videos
# power
# mels
# chroma
# mfcc


frame_size = 4096
hop_size = 512
fft_window = 512
signal_duration = 14


In [3]:
# Execution
filename = utils.find_file_by_upload_id(upload_id, upload_dir)
print(filename)

Upload couldn't be found
Upload couldn't be found
/mnt/g/uploads/Seek & Destroy (Remastered)-003.mp3


In [4]:
import librosa as li
import utils

# Execution
signal, sr = utils.extract_signal_for_inference(filename)

# Storing transformed signal
song_location = utils.generate_audio_from_frames(f"generated_track_{upload_id}", signal, sr, artifacts_dir)

# Frames indexes for generation of data
frames = utils.split_signal_to_frame_indexes(frame_size, hop_size, signal)

In [5]:
def extract_mels_and_generate_artifacts():
    mels = utils.extract_mels(frames, signal)
    utils.generate_mel_spec_images(upload_id, mels, artifacts_dir)
    utils.generate_video_from_mel_spec_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
    return mels

def extract_power_spectrograms_and_generate_artifacts():
    power_spectrograms = utils.extract_power_spectrograms(frames, signal)
    utils.generate_power_spectr_spec_images(upload_id, power_spectrograms, artifacts_dir)
    utils.generate_video_from_power_spetr_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
    return power_spectrograms

def extract_mfcc_and_generate_artifacts():
    mfcc = utils.extract_mfccs(frames, signal)
    utils.generate_mfcc_images(upload_id, mfcc, artifacts_dir)
    utils.generate_video_from_mfcc_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
    return mfcc

mels = extract_mels_and_generate_artifacts()
power_spectrograms = extract_power_spectrograms_and_generate_artifacts()
mfcc = extract_mfcc_and_generate_artifacts()

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [6]:
stacked = utils.stack_mfcc_power_spectro_mel_spectro(mfcc, mels, power_spectrograms)

In [7]:
stacked.shape

(3, 75, 20, 10)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np



def make_predictions_return_percentages(model, stacked):
    x = np.array(stacked).astype(np.float32)
    normalized_x = (x - x.min()) / (x.max() - x.min())
    
    flattened = torch.from_numpy(x).reshape(-1)
    output = model(flattened)
    percentages = F.softmax(output, dim=0) * 100
    return percentages

In [9]:
torch.set_printoptions(precision=4, sci_mode=False)



model = utils.MusicModel()
model.load_state_dict(torch.load(f'{models_dir}{model_filename}'))
model.eval()

pred = make_predictions_return_percentages(model, stacked)

pred

tensor([  0., 100.,   0.,   0.,   0.], grad_fn=<MulBackward0>)